# Nikand: free AI video clips (Wan 2.1)

Ye notebook free open-source model **Wan 2.1 (1.3B)** se website ke liye 4 asli jaisi clips banata hai:

1. Gande kapde washing machine me udte hue
2. Powder machine me girta hua
3. Machine me jhaag ke saath kapde ghoomte hue
4. Saaf kapde tar pe lehrate hue

**Kaise chalayein**
1. Upar menu me **Runtime → Change runtime type → T4 GPU** chuniye, Save.
2. Phir **Runtime → Run all** dabaiye.
3. Har clip me lagbhag 10-25 minute lagte hain. Tab band mat kijiye.
4. End me `nikand_clips.zip` apne aap download hoga. Wo file Claude ko bhej dijiye.

Pouch (packet) AI se nahi banwaya, kyunki AI uske akshar bigaad deta hai. Packet website pe apna 3D wala hi rahega.

In [ ]:
# 1) GPU check
import torch, subprocess
assert torch.cuda.is_available(), "GPU nahi mila. Runtime > Change runtime type > T4 GPU chuniye, phir dobara Run all."
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)

In [ ]:
# 2) Install libraries (2-3 minute)
!pip -q install -U diffusers transformers accelerate ftfy sentencepiece imageio imageio-ffmpeg

In [ ]:
# 3) Scenes: yahan prompt badal sakte hain
SCENES = {
    "1_clothes_into_machine": (
        "Photorealistic detergent commercial. A modern white front-load washing machine stands in a bright, clean Indian home "
        "laundry room, its round glass door wide open. Dirty everyday clothes with mud and food stains, a white shirt, blue jeans, "
        "a pink towel and a yellow t-shirt, fly through the air in slow motion one after another and drop into the drum. "
        "Soft morning daylight, shallow depth of field, steady camera, cinematic, high detail."
    ),
    "2_powder_pouring": (
        "Photorealistic close-up detergent commercial. A hand tilts a plain glossy blue plastic pouch over the open round door of a "
        "white front-load washing machine filled with clothes. White detergent powder with small blue granules pours out in a smooth "
        "stream into the drum, tiny sparkling particles in the air, slow motion, bright studio lighting, high detail."
    ),
    "3_wash_foam": (
        "Photorealistic close-up through the round glass door of a white front-load washing machine. Colourful clothes tumble in "
        "clear soapy water, thick white foam and bubbles swirl as the drum spins, water droplets on the glass, chrome door ring, "
        "bright clean lighting, detergent commercial, high detail."
    ),
    "4_clothes_on_line": (
        "Photorealistic detergent commercial. Fresh clean clothes, a bright white shirt, a light blue shirt, a pink towel, a yellow "
        "t-shirt and blue jeans, hang with wooden pegs on a clothesline on a sunny rooftop and gently wave in the breeze. Clear blue "
        "sky, crisp sunlight, fabric looks soft and spotless, static camera, slow motion, high detail."
    ),
}
NEGATIVE = (
    "cartoon, 3d render, cgi, illustration, anime, text, letters, logo, watermark, subtitles, overexposed, static image, "
    "blurred details, worst quality, low quality, jpeg artifacts, ugly, deformed, extra fingers, poorly drawn hands, "
    "messy background, people walking in background"
)
WIDTH, HEIGHT = 832, 480   # 16:9. Mobile ke liye 480, 832 kar sakte hain
NUM_FRAMES = 81            # ~5 second @ 16 fps
STEPS = 30
SEED = 42

In [ ]:
# 4) Prompt ko text-embedding me badalna (text encoder bada hai, isliye use pehle chala kar memory se hata dete hain)
import gc, torch
from diffusers import WanPipeline
from transformers import UMT5EncoderModel

MODEL_ID = "Wan-AI/Wan2.1-T2V-1.3B-Diffusers"
major, _ = torch.cuda.get_device_capability()
DTYPE = torch.bfloat16 if major >= 8 else torch.float16   # T4 = float16
print("dtype:", DTYPE)

# seedha GPU par load (Colab ki CPU RAM kam hai)
te = UMT5EncoderModel.from_pretrained(MODEL_ID, subfolder="text_encoder", torch_dtype=torch.bfloat16, device_map="cuda")
enc = WanPipeline.from_pretrained(MODEL_ID, text_encoder=te, transformer=None, vae=None, torch_dtype=torch.bfloat16)
EMBEDS = {}
with torch.no_grad():
    for name, prompt in SCENES.items():
        pos, neg = enc.encode_prompt(prompt=prompt, negative_prompt=NEGATIVE, do_classifier_free_guidance=True, max_sequence_length=512, device="cuda")
        EMBEDS[name] = (pos.to("cpu", DTYPE), neg.to("cpu", DTYPE))
        print("encoded:", name)
del enc, te; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# 5) Video model load
from diffusers import AutoencoderKLWan
vae = AutoencoderKLWan.from_pretrained(MODEL_ID, subfolder="vae", torch_dtype=torch.float32)
pipe = WanPipeline.from_pretrained(MODEL_ID, vae=vae, text_encoder=None, tokenizer=None, torch_dtype=DTYPE)
pipe.to("cuda")
print("ready")

In [ ]:
# 6) Clips banana (har clip 10-25 min)
import os, time
from diffusers.utils import export_to_video
from IPython.display import HTML, display
from base64 import b64encode

os.makedirs("nikand_clips", exist_ok=True)
for name, (pos, neg) in EMBEDS.items():
    out = f"nikand_clips/{name}.mp4"
    if os.path.exists(out):
        print("pehle se bani hai:", out); continue
    t0 = time.time()
    frames = pipe(
        prompt_embeds=pos.to("cuda"), negative_prompt_embeds=neg.to("cuda"),
        height=HEIGHT, width=WIDTH, num_frames=NUM_FRAMES, num_inference_steps=STEPS,
        guidance_scale=5.0, generator=torch.Generator("cuda").manual_seed(SEED),
    ).frames[0]
    export_to_video(frames, out, fps=16)
    print(f"{out} bani ({(time.time() - t0) / 60:.1f} min)")
    data = b64encode(open(out, "rb").read()).decode()
    display(HTML(f'<video width="416" controls loop autoplay muted src="data:video/mp4;base64,{data}"></video>'))
    torch.cuda.empty_cache()

In [ ]:
# 7) Zip karke download
import shutil
from google.colab import files
shutil.make_archive("nikand_clips", "zip", "nikand_clips")
files.download("nikand_clips.zip")

**Agar error aaye**
- `CUDA out of memory`: Runtime → Restart session, phir Run all. Ya cell 3 me `NUM_FRAMES = 49` kar dijiye.
- Free GPU ka time khatam ho jaye: kuch ghante baad dobara chalaiye. Jo clips ban chuki hain wo dobara nahi banengi agar session wahi hai, warna pehle bani clips download kar lijiye (left side ke Files icon me `nikand_clips` folder).
- Clip pasand na aaye: cell 3 me `SEED` badal kar (jaise 7, 123) sirf cell 6 aur 7 dobara chalaiye.